<a href="https://colab.research.google.com/github/lidwaa/cinetrack/blob/main/Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
SCRIPT DE DÉBOGAGE — PIPELINE PDF → OCR
=========================================
Pour chaque PDF échoué, ce script inspecte et trace chaque micro-étape :

  ÉTAPE 1 — Lecture du fichier PDF
  ÉTAPE 2 — Conversion PDF → PNG (pdf2image, 300 DPI)
  ÉTAPE 3 — Inspection du PNG produit (taille, luminosité, rotation)
  ÉTAPE 4 — Encodage PNG → base64
  ÉTAPE 5 — Appel Mistral OCR
  ÉTAPE 6 — Analyse de la réponse OCR

Pour chaque PDF, le script génère dans un dossier dédié :
  - page_originale.png       : le PNG produit par pdf2image (ce que vous envoyez)
  - page_annotee.png         : même image avec infos debug superposées
  - debug_log.txt            : log détaillé de toutes les étapes
  - ocr_response_raw.txt     : réponse brute de Mistral OCR

Un rapport global CSV est aussi produit à la fin.

Dépendances :
    pip install pdf2image pymupdf pillow pandas requests

Usage :
    python debug_pipeline.py --failed liste_76_pdfs.txt
"""

import argparse
import base64
import io
import json
import logging
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import fitz  # pymupdf
import pandas as pd
import requests
from PIL import Image, ImageDraw, ImageFont, ImageStat

# ─────────────────────────────────────────────
# ⚙️  CONFIGURATION — ADAPTEZ CES VALEURS
# ─────────────────────────────────────────────

MISTRAL_OCR_URL = "http://votre-serveur-interne/v1/ocr"
MISTRAL_API_KEY  = "votre-cle-api"
REQUEST_TIMEOUT  = 60   # secondes
DPI              = 300  # votre DPI actuel

# Dossier de sortie pour tous les fichiers de debug
OUTPUT_DIR = Path("debug_output")

# Seuil en dessous duquel on considère la réponse OCR comme "vide"
MIN_OCR_CHARS = 50

# ─────────────────────────────────────────────
# Utilitaires
# ─────────────────────────────────────────────

def setup_logger(log_path: Path) -> logging.Logger:
    logger = logging.getLogger(str(log_path))
    logger.setLevel(logging.DEBUG)
    logger.handlers = []
    # Fichier
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(fh)
    # Console
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("  %(message)s"))
    logger.addHandler(ch)
    return logger


def annotate_image(img: Image.Image, annotations: list[str]) -> Image.Image:
    """Superpose des infos de debug en rouge sur le PNG."""
    annotated = img.convert("RGB").copy()
    draw = ImageDraw.Draw(annotated)
    # Fond semi-opaque pour lisibilité
    margin, line_h, font_size = 10, 22, 18
    box_h = len(annotations) * line_h + margin * 2
    draw.rectangle([0, 0, annotated.width, box_h], fill=(0, 0, 0, 180))
    for i, text in enumerate(annotations):
        draw.text((margin, margin + i * line_h), text, fill=(255, 80, 80))
    return annotated


# ─────────────────────────────────────────────
# ÉTAPE 1 — Lecture PDF
# ─────────────────────────────────────────────

def step1_read_pdf(pdf_path: str, log) -> dict:
    log.info("━━ ÉTAPE 1 : Lecture PDF ━━")
    info = {
        "ok": False,
        "file_size_kb": None,
        "num_pages": None,
        "is_encrypted": None,
        "pdf_version": None,
        "page0_rotation": None,
        "page0_width_pt": None,
        "page0_height_pt": None,
        "has_native_text": None,
        "native_text_len": 0,
        "error": None,
    }
    try:
        size_kb = Path(pdf_path).stat().st_size / 1024
        info["file_size_kb"] = round(size_kb, 2)
        log.info(f"  Taille fichier    : {size_kb:.1f} KB")

        doc = fitz.open(pdf_path)
        info["num_pages"]    = doc.page_count
        info["is_encrypted"] = doc.is_encrypted
        info["pdf_version"]  = doc.pdf_version()
        log.info(f"  Version PDF       : {info['pdf_version']}")
        log.info(f"  Nombre de pages   : {info['num_pages']}")
        log.info(f"  Chiffré           : {info['is_encrypted']}")

        if doc.page_count > 0:
            page = doc[0]
            info["page0_rotation"]  = page.rotation
            info["page0_width_pt"]  = round(page.rect.width, 2)
            info["page0_height_pt"] = round(page.rect.height, 2)
            log.info(f"  Dimensions page 0 : {info['page0_width_pt']} x {info['page0_height_pt']} pts")
            log.info(f"  Rotation page 0   : {info['page0_rotation']}°")

            text = page.get_text("text").strip()
            info["native_text_len"] = len(text)
            info["has_native_text"] = len(text) > 20
            log.info(f"  Texte natif       : {len(text)} caractères {'⚠️  (PDF natif, pas scanné !)' if info['has_native_text'] else ''}")

        doc.close()
        info["ok"] = True
        log.info("  ✅ Étape 1 OK")
    except Exception as e:
        info["error"] = str(e)
        log.error(f"  ❌ Étape 1 ÉCHOUÉE : {e}")
    return info


# ─────────────────────────────────────────────
# ÉTAPE 2 — Conversion PDF → PNG
# ─────────────────────────────────────────────

def step2_convert_png(pdf_path: str, out_dir: Path, log) -> dict:
    log.info("━━ ÉTAPE 2 : Conversion PDF → PNG (pdf2image) ━━")
    info = {
        "ok": False,
        "dpi_used": DPI,
        "num_images_returned": 0,
        "error": None,
        "image": None,
    }
    try:
        from pdf2image import convert_from_path
        log.info(f"  DPI               : {DPI}")
        log.info(f"  Pages converties  : 1 (première page uniquement)")

        images = convert_from_path(
            pdf_path,
            dpi=DPI,
            first_page=1,
            last_page=1,
            fmt="PNG",
        )
        info["num_images_returned"] = len(images)
        log.info(f"  Images retournées : {len(images)}")

        if len(images) == 0:
            info["error"] = "convert_from_path a retourné 0 image"
            log.error(f"  ❌ Étape 2 ÉCHOUÉE : liste vide retournée par pdf2image")
            return info

        img = images[0]
        info["image"] = img
        info["ok"] = True

        # Sauvegarde du PNG brut pour inspection visuelle
        png_path = out_dir / "page_originale.png"
        img.save(png_path, format="PNG")
        log.info(f"  PNG sauvegardé    : {png_path}")
        log.info("  ✅ Étape 2 OK")
    except Exception as e:
        info["error"] = str(e)
        log.error(f"  ❌ Étape 2 ÉCHOUÉE : {e}")
    return info


# ─────────────────────────────────────────────
# ÉTAPE 3 — Inspection du PNG
# ─────────────────────────────────────────────

def step3_inspect_png(img: Image.Image, out_dir: Path, log) -> dict:
    log.info("━━ ÉTAPE 3 : Inspection du PNG ━━")
    info = {
        "ok": False,
        "width_px": None,
        "height_px": None,
        "mode": None,
        "size_kb": None,
        "mean_brightness": None,
        "is_blank": None,
        "is_very_dark": None,
        "stddev_brightness": None,
        "aspect_ratio": None,
        "likely_rotated": None,
        "error": None,
    }
    try:
        info["width_px"]  = img.width
        info["height_px"] = img.height
        info["mode"]      = img.mode
        info["aspect_ratio"] = round(img.width / img.height, 3)

        log.info(f"  Dimensions        : {img.width} x {img.height} px")
        log.info(f"  Mode couleur      : {img.mode}")
        log.info(f"  Ratio W/H         : {info['aspect_ratio']}")

        # Taille en mémoire
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        size_kb = len(buf.getvalue()) / 1024
        info["size_kb"] = round(size_kb, 2)
        log.info(f"  Taille PNG        : {size_kb:.1f} KB")

        # Luminosité
        gray = img.convert("L")
        stat = ImageStat.Stat(gray)
        mean = stat.mean[0]
        stddev = stat.stddev[0]
        info["mean_brightness"]   = round(mean, 2)
        info["stddev_brightness"] = round(stddev, 2)
        info["is_blank"]          = mean > 248               # quasi-blanc
        info["is_very_dark"]      = mean < 10                # quasi-noir
        log.info(f"  Luminosité moy.   : {mean:.1f} / 255 {'⚠️  PAGE BLANCHE !' if info['is_blank'] else '⚠️  PAGE NOIRE !' if info['is_very_dark'] else '✅'}")
        log.info(f"  Écart-type lum.   : {stddev:.1f} {'⚠️  Très faible contraste' if stddev < 10 else ''}")

        # Rotation probable : une carte grise est en paysage (W > H)
        # Si W < H → probablement portrait alors que ça devrait être paysage
        info["likely_rotated"] = img.width < img.height
        if info["likely_rotated"]:
            log.warning(f"  ⚠️  Image en PORTRAIT ({img.width}x{img.height}) — carte grise attendue en PAYSAGE, rotation possible !")
        else:
            log.info(f"  Orientation       : Paysage ✅")

        # Image annotée pour inspection visuelle
        annotations = [
            f"Dims: {img.width}x{img.height}px  |  {size_kb:.0f} KB",
            f"Luminosité moy: {mean:.1f}  |  Écart-type: {stddev:.1f}",
            f"{'⚠️  PAGE BLANCHE' if info['is_blank'] else '⚠️  PAGE NOIRE' if info['is_very_dark'] else 'Luminosité OK'}",
            f"{'⚠️  ROTATION SUSPECTE (portrait)' if info['likely_rotated'] else 'Orientation OK (paysage)'}",
        ]
        annotated = annotate_image(img, annotations)
        annotated.save(out_dir / "page_annotee.png", format="PNG")
        log.info(f"  PNG annoté        : {out_dir / 'page_annotee.png'}")

        info["ok"] = True
        log.info("  ✅ Étape 3 OK")
    except Exception as e:
        info["error"] = str(e)
        log.error(f"  ❌ Étape 3 ÉCHOUÉE : {e}")
    return info


# ─────────────────────────────────────────────
# ÉTAPE 4 — Encodage PNG → base64
# ─────────────────────────────────────────────

def step4_encode_b64(img: Image.Image, log) -> dict:
    log.info("━━ ÉTAPE 4 : Encodage PNG → base64 ━━")
    info = {
        "ok": False,
        "b64_length": None,
        "b64_size_kb": None,
        "b64_preview": None,
        "b64": None,
        "error": None,
    }
    try:
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        raw_bytes = buf.getvalue()
        b64 = base64.b64encode(raw_bytes).decode("utf-8")

        info["b64_length"]  = len(b64)
        info["b64_size_kb"] = round(len(b64) / 1024, 2)
        info["b64_preview"] = b64[:80] + "..."
        info["b64"]         = b64

        log.info(f"  Longueur base64   : {len(b64)} caractères")
        log.info(f"  Taille base64     : {info['b64_size_kb']} KB")
        log.info(f"  Début base64      : {info['b64_preview']}")

        if len(b64) < 1000:
            log.warning(f"  ⚠️  base64 très court ({len(b64)} chars) — image probablement vide ou corrompue")

        info["ok"] = True
        log.info("  ✅ Étape 4 OK")
    except Exception as e:
        info["error"] = str(e)
        log.error(f"  ❌ Étape 4 ÉCHOUÉE : {e}")
    return info


# ─────────────────────────────────────────────
# ÉTAPE 5 — Appel Mistral OCR
# ─────────────────────────────────────────────

def step5_call_ocr(b64: str, filename: str, out_dir: Path, log) -> dict:
    log.info("━━ ÉTAPE 5 : Appel Mistral OCR ━━")
    info = {
        "ok": False,
        "http_status": None,
        "response_time_s": None,
        "raw_response": None,
        "error": None,
    }
    # ⚠️  Adaptez le payload à votre API interne
    payload = {
        "image": b64,
        "filename": filename,
    }
    headers = {"Authorization": f"Bearer {MISTRAL_API_KEY}"}
    log.info(f"  URL               : {MISTRAL_OCR_URL}")
    log.info(f"  Timeout           : {REQUEST_TIMEOUT}s")

    try:
        t0 = time.time()
        resp = requests.post(
            MISTRAL_OCR_URL,
            json=payload,
            headers=headers,
            timeout=REQUEST_TIMEOUT,
        )
        elapsed = round(time.time() - t0, 2)
        info["http_status"]      = resp.status_code
        info["response_time_s"]  = elapsed

        log.info(f"  HTTP status       : {resp.status_code}")
        log.info(f"  Temps réponse     : {elapsed}s")

        if resp.status_code != 200:
            info["error"] = f"HTTP {resp.status_code} : {resp.text[:300]}"
            log.error(f"  ❌ Étape 5 ÉCHOUÉE : {info['error']}")
            (out_dir / "ocr_response_raw.txt").write_text(
                f"HTTP {resp.status_code}\n{resp.text}", encoding="utf-8"
            )
            return info

        raw = resp.text
        info["raw_response"] = raw
        (out_dir / "ocr_response_raw.txt").write_text(raw, encoding="utf-8")
        log.info(f"  Taille réponse    : {len(raw)} caractères")
        log.info(f"  Début réponse     : {raw[:200].strip()}")

        info["ok"] = True
        log.info("  ✅ Étape 5 OK")
    except requests.exceptions.Timeout:
        info["error"] = f"Timeout après {REQUEST_TIMEOUT}s"
        log.error(f"  ❌ Étape 5 TIMEOUT — le serveur OCR n'a pas répondu en {REQUEST_TIMEOUT}s")
    except Exception as e:
        info["error"] = str(e)
        log.error(f"  ❌ Étape 5 ÉCHOUÉE : {e}")
    return info


# ─────────────────────────────────────────────
# ÉTAPE 6 — Analyse réponse OCR
# ─────────────────────────────────────────────

def step6_analyze_ocr(raw_response: str, log) -> dict:
    log.info("━━ ÉTAPE 6 : Analyse réponse OCR ━━")
    info = {
        "ok": False,
        "extracted_text": None,
        "text_length": 0,
        "is_empty": True,
        "is_truncated_suspect": False,
        "diagnosis": None,
        "error": None,
    }
    try:
        # Tenter de parser le JSON de réponse
        # ⚠️  Adaptez selon la structure de réponse de votre API
        try:
            data = json.loads(raw_response)
            text = (
                data.get("text") or
                data.get("content") or
                data.get("result") or
                data.get("output") or
                ""
            )
        except json.JSONDecodeError:
            # La réponse est peut-être directement du texte brut
            text = raw_response
            log.warning("  ⚠️  Réponse non-JSON, traitement comme texte brut")

        text = text.strip() if isinstance(text, str) else ""
        info["extracted_text"] = text
        info["text_length"]    = len(text)
        info["is_empty"]       = len(text) < MIN_OCR_CHARS

        log.info(f"  Longueur texte    : {len(text)} caractères")

        if len(text) == 0:
            info["diagnosis"] = "VIDE : OCR n'a retourné aucun texte"
            log.error(f"  ❌ {info['diagnosis']}")
        elif len(text) < MIN_OCR_CHARS:
            info["diagnosis"] = f"TRONQUÉ : seulement {len(text)} chars (seuil={MIN_OCR_CHARS})"
            log.warning(f"  ⚠️  {info['diagnosis']}")
            log.warning(f"  Contenu           : '{text}'")
        else:
            # Vérifier troncature suspecte : texte qui s'arrête en plein milieu
            last_char = text[-1] if text else ""
            info["is_truncated_suspect"] = last_char not in ".,:;)\"'\n " and len(text) > 100
            if info["is_truncated_suspect"]:
                log.warning(f"  ⚠️  Troncature suspecte — dernier char: '{last_char}'")
            info["diagnosis"] = "OK" if not info["is_empty"] else "VIDE"
            log.info(f"  Aperçu texte      : {text[:300].strip()}")
            log.info("  ✅ Étape 6 OK")

        info["ok"] = not info["is_empty"]
    except Exception as e:
        info["error"] = str(e)
        log.error(f"  ❌ Étape 6 ÉCHOUÉE : {e}")
    return info


# ─────────────────────────────────────────────
# Pipeline complet pour 1 PDF
# ─────────────────────────────────────────────

def debug_one_pdf(pdf_path: str) -> dict:
    pdf_name = Path(pdf_path).stem
    out_dir  = OUTPUT_DIR / pdf_name
    out_dir.mkdir(parents=True, exist_ok=True)

    log = setup_logger(out_dir / "debug_log.txt")
    log.info(f"{'='*60}")
    log.info(f"DÉBOGAGE : {Path(pdf_path).name}")
    log.info(f"Dossier  : {out_dir}")
    log.info(f"Date     : {datetime.now().isoformat()}")
    log.info(f"{'='*60}")

    summary = {
        "filename":        Path(pdf_path).name,
        "output_folder":   str(out_dir),
        # Résultats par étape
        "step1_ok": False, "step1_error": None,
        "step2_ok": False, "step2_error": None,
        "step3_ok": False, "step3_error": None,
        "step4_ok": False, "step4_error": None,
        "step5_ok": False, "step5_http_status": None, "step5_error": None,
        "step6_ok": False, "step6_diagnosis": None,
        # Métriques clés pour repérer le problème
        "file_size_kb":     None,
        "num_pages":        None,
        "page0_rotation":   None,
        "has_native_text":  None,
        "png_width_px":     None,
        "png_height_px":    None,
        "png_size_kb":      None,
        "png_brightness":   None,
        "png_is_blank":     None,
        "png_likely_rotated": None,
        "b64_size_kb":      None,
        "ocr_response_time_s": None,
        "ocr_text_length":  None,
        "failure_step":     None,  # Première étape qui échoue
    }

    # ── Étape 1 ──────────────────────────────────────────────────────────
    s1 = step1_read_pdf(pdf_path, log)
    summary.update({
        "step1_ok": s1["ok"], "step1_error": s1["error"],
        "file_size_kb": s1["file_size_kb"], "num_pages": s1["num_pages"],
        "page0_rotation": s1["page0_rotation"], "has_native_text": s1["has_native_text"],
    })
    if not s1["ok"]:
        summary["failure_step"] = "ÉTAPE 1 — Lecture PDF"
        log.error(f"\n🔴 PROBLÈME DÉTECTÉ À L'ÉTAPE 1 : {s1['error']}")
        return summary

    # ── Étape 2 ──────────────────────────────────────────────────────────
    s2 = step2_convert_png(pdf_path, out_dir, log)
    summary.update({"step2_ok": s2["ok"], "step2_error": s2["error"]})
    if not s2["ok"]:
        summary["failure_step"] = "ÉTAPE 2 — Conversion PDF→PNG"
        log.error(f"\n🔴 PROBLÈME DÉTECTÉ À L'ÉTAPE 2 : {s2['error']}")
        return summary

    img = s2["image"]

    # ── Étape 3 ──────────────────────────────────────────────────────────
    s3 = step3_inspect_png(img, out_dir, log)
    summary.update({
        "step3_ok": s3["ok"], "step3_error": s3["error"],
        "png_width_px": s3["width_px"], "png_height_px": s3["height_px"],
        "png_size_kb": s3["size_kb"], "png_brightness": s3["mean_brightness"],
        "png_is_blank": s3["is_blank"], "png_likely_rotated": s3["likely_rotated"],
    })
    # On continue même si problème détecté (pas un crash, juste une anomalie)
    if s3["is_blank"]:
        summary["failure_step"] = summary["failure_step"] or "ÉTAPE 3 — PNG blanc/noir"
        log.error("\n🔴 PROBLÈME DÉTECTÉ À L'ÉTAPE 3 : page blanche ou noire")

    # ── Étape 4 ──────────────────────────────────────────────────────────
    s4 = step4_encode_b64(img, log)
    summary.update({
        "step4_ok": s4["ok"], "step4_error": s4["error"],
        "b64_size_kb": s4["b64_size_kb"],
    })
    if not s4["ok"]:
        summary["failure_step"] = summary["failure_step"] or "ÉTAPE 4 — Encodage base64"
        log.error(f"\n🔴 PROBLÈME DÉTECTÉ À L'ÉTAPE 4 : {s4['error']}")
        return summary

    # ── Étape 5 ──────────────────────────────────────────────────────────
    s5 = step5_call_ocr(s4["b64"], Path(pdf_path).name, out_dir, log)
    summary.update({
        "step5_ok": s5["ok"], "step5_http_status": s5["http_status"],
        "step5_error": s5["error"], "ocr_response_time_s": s5["response_time_s"],
    })
    if not s5["ok"]:
        summary["failure_step"] = summary["failure_step"] or "ÉTAPE 5 — Appel OCR"
        log.error(f"\n🔴 PROBLÈME DÉTECTÉ À L'ÉTAPE 5 : {s5['error']}")
        return summary

    # ── Étape 6 ──────────────────────────────────────────────────────────
    s6 = step6_analyze_ocr(s5["raw_response"], log)
    summary.update({
        "step6_ok": s6["ok"], "step6_diagnosis": s6["diagnosis"],
        "ocr_text_length": s6["text_length"],
    })
    if not s6["ok"]:
        summary["failure_step"] = summary["failure_step"] or "ÉTAPE 6 — Réponse OCR vide/tronquée"
        log.error(f"\n🔴 PROBLÈME DÉTECTÉ À L'ÉTAPE 6 : {s6['diagnosis']}")

    # ── Verdict final ─────────────────────────────────────────────────────
    if not summary["failure_step"]:
        log.info(f"\n✅ Aucun problème détecté dans le pipeline pour ce PDF")
        summary["failure_step"] = "AUCUN — pipeline OK"

    log.info(f"\n{'='*60}")
    log.info(f"VERDICT : {summary['failure_step']}")
    log.info(f"{'='*60}")

    return summary


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Débogage pipeline PDF → OCR")
    parser.add_argument(
        "--failed", required=True,
        help="Fichier texte avec la liste des PDFs échoués (un chemin par ligne)"
    )
    parser.add_argument(
        "--output", default="debug_rapport.csv",
        help="Rapport CSV de synthèse (défaut: debug_rapport.csv)"
    )
    args = parser.parse_args()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    with open(args.failed, "r", encoding="utf-8") as f:
        failed_paths = [l.strip() for l in f if l.strip()]

    print(f"\n{'='*60}")
    print(f"  DÉBOGAGE PIPELINE PDF → OCR")
    print(f"  PDFs à analyser : {len(failed_paths)}")
    print(f"  Dossier output  : {OUTPUT_DIR}/")
    print(f"{'='*60}\n")

    rows = []
    for i, pdf_path in enumerate(failed_paths, 1):
        print(f"\n[{i}/{len(failed_paths)}] {Path(pdf_path).name}")
        summary = debug_one_pdf(pdf_path)
        rows.append(summary)

    # ── Rapport CSV ───────────────────────────────────────────────────────
    df = pd.DataFrame(rows)
    df.to_csv(args.output, index=False, encoding="utf-8-sig")

    # ── Synthèse console ──────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("  SYNTHÈSE")
    print(f"{'='*60}")
    print(f"\n  Étape de premier échec :")
    for step, count in df["failure_step"].value_counts().items():
        print(f"    {count:3d} PDF(s)  →  {step}")

    print(f"\n  Pages blanches détectées  : {df['png_is_blank'].sum()}")
    print(f"  Rotations suspectes       : {df['png_likely_rotated'].sum()}")
    print(f"  PDFs avec texte natif     : {df['has_native_text'].sum()}")
    if df["ocr_response_time_s"].notna().any():
        print(f"  Temps OCR moy.            : {df['ocr_response_time_s'].mean():.1f}s")
        print(f"  Temps OCR max.            : {df['ocr_response_time_s'].max():.1f}s")

    print(f"\n  Rapport CSV   : {args.output}")
    print(f"  Logs/PNGs     : {OUTPUT_DIR}/<nom_pdf>/")
    print(f"     ├── debug_log.txt        (log détaillé)")
    print(f"     ├── page_originale.png   (ce qui est envoyé à l'OCR)")
    print(f"     ├── page_annotee.png     (image + infos debug)")
    print(f"     └── ocr_response_raw.txt (réponse brute Mistral OCR)")
    print(f"\n{'='*60}\n")


if __name__ == "__main__":
    main()

In [ ]:
"""
APPROCHE A — Détection PDF natif + extraction texte directe
============================================================
Pour chaque PDF de la liste des 76 échoués :
  - Teste si le PDF contient du texte sélectionnable (PyMuPDF)
  - Si oui  → extrait le texte directement, envoie à Mistral Medium
  - Si non  → signale que le PDF est un scan (pipeline OCR nécessaire)

Produit :
  - approche_a_resultats.xlsx          : résultats parsés pour les PDFs natifs
  - approche_a_rapport.csv             : rapport par PDF (natif / scan / erreur)
  - approche_a_texte_brut/             : textes extraits bruts (un .txt par PDF)
  - liste_scans_pour_approche_b.txt    : noms des scans à traiter avec B

Dépendances :
    pip install pymupdf pandas openpyxl requests

Usage :
    python approche_a.py
"""

import json
import requests
from pathlib import Path

import fitz  # pymupdf
import pandas as pd

# ─────────────────────────────────────────────
# ⚙️  CONFIGURATION — ADAPTEZ CES VALEURS
# ─────────────────────────────────────────────

# Dossier contenant tous les PDFs
PDF_DIR = Path("/chemin/vers/dossier/pdfs")

# Fichier texte avec les noms des PDFs échoués (un nom par ligne, ex: doc123.pdf)
FAILED_LIST = Path("/chemin/vers/liste_76_noms.txt")

# Mettre False pour appeler Mistral Medium sur les PDFs natifs trouvés
# Mettre True pour faire uniquement l'extraction texte sans appel API
NO_PARSE = False

MISTRAL_MEDIUM_URL = "http://votre-serveur-interne/v1/chat"
MISTRAL_API_KEY    = "votre-cle-api"
REQUEST_TIMEOUT    = 60

# Seuil : nb de caractères minimum pour considérer qu'une page a du texte utile
MIN_TEXT_CHARS = 50

# Fichiers de sortie
OUTPUT_EXCEL    = Path("approche_a_resultats.xlsx")
OUTPUT_RAPPORT  = Path("approche_a_rapport.csv")
OUTPUT_BRUT_DIR = Path("approche_a_texte_brut")
OUTPUT_SCANS    = Path("liste_scans_pour_approche_b.txt")


# ─────────────────────────────────────────────
# Extraction texte natif via PyMuPDF
# ─────────────────────────────────────────────

def extraire_texte_natif(pdf_path: str) -> dict:
    result = {
        "est_natif":      False,
        "nb_pages":       0,
        "texte":          "",
        "chars_par_page": [],
        "erreur":         None,
    }
    try:
        doc = fitz.open(pdf_path)
        result["nb_pages"] = doc.page_count

        texte_total = ""
        for page in doc:
            texte_page = page.get_text("text").strip()
            result["chars_par_page"].append(len(texte_page))
            texte_total += texte_page + "\n"

        doc.close()

        texte_total = texte_total.strip()
        result["texte"]     = texte_total
        result["est_natif"] = any(c >= MIN_TEXT_CHARS for c in result["chars_par_page"])

    except Exception as e:
        result["erreur"] = str(e)

    return result


# ─────────────────────────────────────────────
# Appel Mistral Medium pour parser le texte
# ─────────────────────────────────────────────

SYSTEM_PROMPT = """Tu es un expert en extraction de données de cartes grises espagnoles (Permiso de Circulación).
À partir du texte brut fourni, extrais les informations et retourne UNIQUEMENT un JSON valide,
sans markdown, sans backticks, sans commentaire. Structure attendue :
{
  "numero_bastidor": null,
  "matricula": null,
  "marca": null,
  "modelo": null,
  "fecha_primera_matriculacion": null,
  "titular_nombre": null,
  "titular_dni": null,
  "direccion": null,
  "municipio": null,
  "provincia": null,
  "potencia_kw": null,
  "cilindrada": null,
  "masa_maxima": null,
  "categoria": null,
  "numero_plazas": null
}
Si une valeur est introuvable dans le texte, mets null."""


def parser_avec_mistral_medium(texte: str) -> dict:
    payload = {
        "model": "mistral-medium",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": texte},
        ],
        "temperature": 0,
    }
    headers = {"Authorization": f"Bearer {MISTRAL_API_KEY}"}
    try:
        resp = requests.post(
            MISTRAL_MEDIUM_URL,
            json=payload,
            headers=headers,
            timeout=REQUEST_TIMEOUT,
        )
        resp.raise_for_status()
        raw = resp.json()["choices"][0]["message"]["content"].strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"    ⚠️  Mistral Medium erreur : {e}")
        return {}


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    # ── Vérifications ─────────────────────────────────────────────────────
    if not PDF_DIR.is_dir():
        print(f"❌ PDF_DIR introuvable : {PDF_DIR}")
        return
    if not FAILED_LIST.exists():
        print(f"❌ FAILED_LIST introuvable : {FAILED_LIST}")
        return

    OUTPUT_BRUT_DIR.mkdir(exist_ok=True)

    # ── Lecture de la liste ───────────────────────────────────────────────
    noms = [
        l.strip()
        for l in FAILED_LIST.read_text(encoding="utf-8").splitlines()
        if l.strip()
    ]

    # Reconstruction des chemins complets
    chemins = []
    introuvables = []
    for nom in noms:
        chemin = PDF_DIR / nom
        if not chemin.exists():
            print(f"⚠️  Fichier introuvable : {chemin}")
            introuvables.append(nom)
        chemins.append(str(chemin))

    print(f"\n{'='*55}")
    print(f"  APPROCHE A — Détection PDF natif")
    print(f"  Dossier PDFs    : {PDF_DIR}")
    print(f"  PDFs à analyser : {len(chemins)}")
    if introuvables:
        print(f"  ⚠️  Introuvables  : {len(introuvables)}")
    print(f"{'='*55}\n")

    natifs, scans, erreurs = [], [], []
    rows_rapport, rows_resultats = [], []

    for i, chemin in enumerate(chemins, 1):
        nom = Path(chemin).name
        print(f"[{i:3}/{len(chemins)}] {nom}")

        extraction = extraire_texte_natif(chemin)

        row_rapport = {
            "filename":       nom,
            "nb_pages":       extraction["nb_pages"],
            "est_natif":      extraction["est_natif"],
            "total_chars":    len(extraction["texte"]),
            "chars_par_page": str(extraction["chars_par_page"]),
            "erreur":         extraction["erreur"],
        }

        if extraction["erreur"]:
            print(f"         ❌ Erreur lecture : {extraction['erreur']}")
            erreurs.append(nom)
            row_rapport["statut"] = "ERREUR"

        elif not extraction["est_natif"]:
            print(f"         📷 Scan ({extraction['nb_pages']} page(s)) → OCR nécessaire")
            scans.append(nom)
            row_rapport["statut"] = "SCAN"
            if extraction["texte"]:
                (OUTPUT_BRUT_DIR / (Path(chemin).stem + "_brut.txt")).write_text(
                    extraction["texte"], encoding="utf-8"
                )

        else:
            print(f"         ✅ PDF natif — {len(extraction['texte'])} chars extraits")
            natifs.append(nom)
            row_rapport["statut"] = "NATIF"

            (OUTPUT_BRUT_DIR / (Path(chemin).stem + "_brut.txt")).write_text(
                extraction["texte"], encoding="utf-8"
            )

            donnees = {}
            if not NO_PARSE:
                print(f"         → Envoi à Mistral Medium...")
                donnees = parser_avec_mistral_medium(extraction["texte"])

            rows_resultats.append({"filename": nom, "statut": "NATIF", **donnees})

        rows_rapport.append(row_rapport)

    # ── Exports ───────────────────────────────────────────────────────────
    pd.DataFrame(rows_rapport).to_csv(OUTPUT_RAPPORT, index=False, encoding="utf-8-sig")

    if rows_resultats:
        pd.DataFrame(rows_resultats).to_excel(OUTPUT_EXCEL, index=False)

    if scans:
        OUTPUT_SCANS.write_text("\n".join(scans), encoding="utf-8")

    # ── Synthèse console ──────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  RÉSULTATS APPROCHE A")
    print(f"{'='*55}")
    print(f"  PDFs natifs  (texte extrait OK) : {len(natifs)}")
    print(f"  PDFs scans   (OCR nécessaire)   : {len(scans)}")
    print(f"  Erreurs lecture                 : {len(erreurs)}")
    print(f"\n  → Excel résultats  : {OUTPUT_EXCEL}")
    print(f"  → Rapport CSV      : {OUTPUT_RAPPORT}")
    print(f"  → Textes bruts     : {OUTPUT_BRUT_DIR}/")
    if scans:
        print(f"  → Liste scans B    : {OUTPUT_SCANS}")
    print(f"{'='*55}\n")


if __name__ == "__main__":
    main()

In [ ]:
"""
APPROCHE B — Sauvegarde des PNGs intermédiaires pour inspection visuelle
=========================================================================
Reproduit EXACTEMENT votre pipeline actuel (PDF → PNG 300 DPI → base64)
mais sauvegarde le PNG sur disque avant l'appel OCR.

Vous pouvez ensuite ouvrir les images à la main et voir immédiatement
si le problème vient de la conversion (image blanche, tournée, illisible)
ou de Mistral OCR lui-même.

Produit pour chaque PDF dans approche_b_pngs/<nom_pdf>/ :
  - page.png          : le PNG envoyé à Mistral OCR (pipeline exact)
  - ocr_reponse.txt   : la réponse brute de Mistral OCR
  - info.txt          : dimensions, luminosité, statut OCR

Produit aussi :
  - approche_b_rapport.csv : tableau récapitulatif de tous les PDFs

Dépendances :
    pip install pdf2image pillow pandas requests
    + poppler installé (apt install poppler-utils)

Usage :
    python approche_b.py
"""

import base64
import io
import time
import requests
from pathlib import Path

import pandas as pd
from pdf2image import convert_from_path
from PIL import Image, ImageStat

# ─────────────────────────────────────────────
# ⚙️  CONFIGURATION — ADAPTEZ CES VALEURS
# ─────────────────────────────────────────────

# Dossier contenant tous les PDFs
PDF_DIR = Path("/chemin/vers/dossier/pdfs")

# Fichier texte avec les noms des PDFs échoués (un nom par ligne, ex: doc123.pdf)
FAILED_LIST = Path("/chemin/vers/liste_76_noms.txt")

# Mettre False pour appeler Mistral OCR après la conversion PNG
# Mettre True pour inspecter les PNGs uniquement, sans appel API (plus rapide)
NO_OCR = False

MISTRAL_OCR_URL = "http://votre-serveur-interne/v1/ocr"
MISTRAL_API_KEY  = "votre-cle-api"
REQUEST_TIMEOUT  = 60
DPI              = 300   # votre DPI actuel

MIN_OCR_CHARS = 50

# Fichiers de sortie
OUTPUT_DIR     = Path("approche_b_pngs")
OUTPUT_RAPPORT = Path("approche_b_rapport.csv")


# ─────────────────────────────────────────────
# Étape 1 — Conversion PDF → PNG (votre pipeline)
# ─────────────────────────────────────────────

def convertir_pdf_png(pdf_path: str) -> dict:
    result = {"ok": False, "image": None, "erreur": None}
    try:
        images = convert_from_path(
            pdf_path,
            dpi=DPI,
            first_page=1,
            last_page=1,
            fmt="PNG",
        )
        if not images:
            result["erreur"] = "pdf2image a retourné une liste vide"
            return result
        result["image"] = images[0]
        result["ok"]    = True
    except Exception as e:
        result["erreur"] = str(e)
    return result


# ─────────────────────────────────────────────
# Étape 2 — Inspection du PNG
# ─────────────────────────────────────────────

def inspecter_png(img: Image.Image) -> dict:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    taille_kb = round(len(buf.getvalue()) / 1024, 2)

    gray      = img.convert("L")
    stat      = ImageStat.Stat(gray)
    luminosite = round(stat.mean[0], 2)
    ecart_type = round(stat.stddev[0], 2)

    return {
        "largeur_px":           img.width,
        "hauteur_px":           img.height,
        "taille_kb":            taille_kb,
        "luminosite":           luminosite,
        "ecart_type":           ecart_type,
        "est_blanc":            luminosite > 248,
        "est_noir":             luminosite < 5,
        "faible_contraste":     ecart_type < 10,
        "probablement_tourne":  img.width < img.height,
    }


# ─────────────────────────────────────────────
# Étape 3 — Encodage base64 (votre pipeline)
# ─────────────────────────────────────────────

def encoder_base64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


# ─────────────────────────────────────────────
# Étape 4 — Appel Mistral OCR (votre pipeline)
# ─────────────────────────────────────────────

def appeler_mistral_ocr(b64: str, filename: str) -> dict:
    result = {
        "ok":          False,
        "http_status": None,
        "temps_s":     None,
        "texte":       "",
        "nb_chars":    0,
        "erreur":      None,
    }
    # ⚠️  Adaptez le payload à votre API interne
    payload = {"image": b64, "filename": filename}
    headers = {"Authorization": f"Bearer {MISTRAL_API_KEY}"}
    try:
        t0 = time.time()
        resp = requests.post(
            MISTRAL_OCR_URL,
            json=payload,
            headers=headers,
            timeout=REQUEST_TIMEOUT,
        )
        result["temps_s"]     = round(time.time() - t0, 2)
        result["http_status"] = resp.status_code
        resp.raise_for_status()

        data  = resp.json()
        texte = data.get("text") or data.get("content") or data.get("result") or ""
        texte = texte.strip()

        result["texte"]    = texte
        result["nb_chars"] = len(texte)
        result["ok"]       = len(texte) >= MIN_OCR_CHARS

    except requests.exceptions.Timeout:
        result["erreur"] = f"Timeout ({REQUEST_TIMEOUT}s)"
    except Exception as e:
        result["erreur"] = str(e)

    return result


# ─────────────────────────────────────────────
# Pipeline complet pour 1 PDF
# ─────────────────────────────────────────────

def traiter_un_pdf(pdf_path: str) -> dict:
    nom     = Path(pdf_path).name
    dossier = OUTPUT_DIR / Path(pdf_path).stem
    dossier.mkdir(parents=True, exist_ok=True)

    row = {
        "filename":             nom,
        "dossier_png":          str(dossier),
        "conversion_ok":        False,
        "conversion_erreur":    None,
        "png_largeur_px":       None,
        "png_hauteur_px":       None,
        "png_taille_kb":        None,
        "png_luminosite":       None,
        "png_ecart_type":       None,
        "png_est_blanc":        None,
        "png_est_noir":         None,
        "png_faible_contraste": None,
        "png_tourne_suspect":   None,
        "ocr_appele":           not NO_OCR,
        "ocr_ok":               None,
        "ocr_http_status":      None,
        "ocr_temps_s":          None,
        "ocr_nb_chars":         None,
        "ocr_erreur":           None,
        "probleme_detecte":     "—",
    }

    # ── Conversion ────────────────────────────────────────────────────────
    conv = convertir_pdf_png(pdf_path)
    row["conversion_ok"]     = conv["ok"]
    row["conversion_erreur"] = conv["erreur"]

    if not conv["ok"]:
        row["probleme_detecte"] = f"CONVERSION ÉCHOUÉE : {conv['erreur']}"
        (dossier / "info.txt").write_text(
            f"ERREUR CONVERSION\n{conv['erreur']}", encoding="utf-8"
        )
        print(f"    ❌ Conversion échouée : {conv['erreur']}")
        return row

    img = conv["image"]

    # ── Sauvegarde PNG ────────────────────────────────────────────────────
    img.save(dossier / "page.png", format="PNG")

    # ── Inspection PNG ────────────────────────────────────────────────────
    info = inspecter_png(img)
    row.update({
        "png_largeur_px":       info["largeur_px"],
        "png_hauteur_px":       info["hauteur_px"],
        "png_taille_kb":        info["taille_kb"],
        "png_luminosite":       info["luminosite"],
        "png_ecart_type":       info["ecart_type"],
        "png_est_blanc":        info["est_blanc"],
        "png_est_noir":         info["est_noir"],
        "png_faible_contraste": info["faible_contraste"],
        "png_tourne_suspect":   info["probablement_tourne"],
    })

    warnings = []
    if info["est_blanc"]:           warnings.append("PAGE BLANCHE")
    if info["est_noir"]:            warnings.append("PAGE NOIRE")
    if info["faible_contraste"]:    warnings.append("CONTRASTE FAIBLE")
    if info["probablement_tourne"]: warnings.append("ROTATION SUSPECTE (portrait)")
    if info["taille_kb"] < 50:      warnings.append(f"PNG TRÈS PETIT ({info['taille_kb']} KB)")

    info_txt = (
        f"Fichier      : {nom}\n"
        f"Dimensions   : {info['largeur_px']} x {info['hauteur_px']} px\n"
        f"Taille PNG   : {info['taille_kb']} KB\n"
        f"Luminosité   : {info['luminosite']} / 255\n"
        f"Écart-type   : {info['ecart_type']} (faible = peu de contraste)\n"
        f"Page blanche : {'OUI ⚠️' if info['est_blanc'] else 'non'}\n"
        f"Page noire   : {'OUI ⚠️' if info['est_noir'] else 'non'}\n"
        f"Faible cont. : {'OUI ⚠️' if info['faible_contraste'] else 'non'}\n"
        f"Rotation ?   : {'OUI ⚠️  (portrait alors qu\'attendu paysage)' if info['probablement_tourne'] else 'non (paysage OK)'}\n"
    )

    if warnings:
        row["probleme_detecte"] = " | ".join(warnings)
        info_txt += f"\n⚠️  PROBLÈMES DÉTECTÉS : {row['probleme_detecte']}\n"
        print(f"    ⚠️  PNG suspect : {row['probleme_detecte']}")
    else:
        info_txt += "\nPNG visuellement correct → problème probablement côté OCR\n"
        print(f"    ✅ PNG OK ({info['largeur_px']}x{info['hauteur_px']}px, "
              f"{info['taille_kb']}KB, lum={info['luminosite']})")

    # ── OCR ───────────────────────────────────────────────────────────────
    if not NO_OCR:
        b64 = encoder_base64(img)
        ocr = appeler_mistral_ocr(b64, nom)

        row.update({
            "ocr_ok":          ocr["ok"],
            "ocr_http_status": ocr["http_status"],
            "ocr_temps_s":     ocr["temps_s"],
            "ocr_nb_chars":    ocr["nb_chars"],
            "ocr_erreur":      ocr["erreur"],
        })

        (dossier / "ocr_reponse.txt").write_text(
            ocr["texte"] or f"[VIDE — erreur: {ocr['erreur']}]",
            encoding="utf-8"
        )

        if not ocr["ok"]:
            diag = ocr["erreur"] or f"Réponse trop courte ({ocr['nb_chars']} chars)"
            if not warnings:
                row["probleme_detecte"] = f"OCR ÉCHOUÉ : {diag}"
            info_txt += f"\nOCR : ÉCHOUÉ — {diag}\n"
            print(f"    ❌ OCR échoué : {diag}")
        else:
            if not warnings:
                row["probleme_detecte"] = "AUCUN — pipeline OK"
            info_txt += f"\nOCR : OK ({ocr['nb_chars']} chars, {ocr['temps_s']}s)\n"
            print(f"    ✅ OCR OK ({ocr['nb_chars']} chars en {ocr['temps_s']}s)")
    else:
        info_txt += "\nOCR : non appelé (NO_OCR = True)\n"

    (dossier / "info.txt").write_text(info_txt, encoding="utf-8")
    return row


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    # ── Vérifications ─────────────────────────────────────────────────────
    if not PDF_DIR.is_dir():
        print(f"❌ PDF_DIR introuvable : {PDF_DIR}")
        return
    if not FAILED_LIST.exists():
        print(f"❌ FAILED_LIST introuvable : {FAILED_LIST}")
        return

    OUTPUT_DIR.mkdir(exist_ok=True)

    # ── Lecture de la liste ───────────────────────────────────────────────
    noms = [
        l.strip()
        for l in FAILED_LIST.read_text(encoding="utf-8").splitlines()
        if l.strip()
    ]

    chemins = []
    introuvables = []
    for nom in noms:
        chemin = PDF_DIR / nom
        if not chemin.exists():
            print(f"⚠️  Fichier introuvable : {chemin}")
            introuvables.append(nom)
        chemins.append(str(chemin))

    mode = "inspection PNG uniquement" if NO_OCR else "PNG + appel OCR"
    print(f"\n{'='*55}")
    print(f"  APPROCHE B — Sauvegarde PNGs")
    print(f"  Dossier PDFs    : {PDF_DIR}")
    print(f"  PDFs à traiter  : {len(chemins)}")
    if introuvables:
        print(f"  ⚠️  Introuvables  : {len(introuvables)}")
    print(f"  Mode            : {mode}")
    print(f"  Dossier sortie  : {OUTPUT_DIR}/")
    print(f"{'='*55}\n")

    rows = []
    for i, chemin in enumerate(chemins, 1):
        print(f"[{i:3}/{len(chemins)}] {Path(chemin).name}")
        row = traiter_un_pdf(chemin)
        rows.append(row)

    # ── Rapport CSV ───────────────────────────────────────────────────────
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_RAPPORT, index=False, encoding="utf-8-sig")

    # ── Synthèse console ──────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  SYNTHÈSE APPROCHE B")
    print(f"{'='*55}")
    print(f"  PNGs blancs/noirs       : {(df['png_est_blanc'] | df['png_est_noir']).sum()}")
    print(f"  Faible contraste        : {df['png_faible_contraste'].sum()}")
    print(f"  Rotation suspecte       : {df['png_tourne_suspect'].sum()}")
    print(f"  Erreurs conversion      : {(~df['conversion_ok']).sum()}")
    if not NO_OCR:
        print(f"  OCR échoués             : {(~df['ocr_ok'].fillna(True)).sum()}")
        print(f"  PNG OK mais OCR échoué  : "
              f"{((~df['png_est_blanc']) & (~df['png_est_noir']) & (~df['ocr_ok'].fillna(True))).sum()}")

    print(f"\n  Rapport CSV  : {OUTPUT_RAPPORT}")
    print(f"  PNGs + infos : {OUTPUT_DIR}/<nom_pdf>/page.png")
    print(f"\n  ➡  Ouvrez les PNGs dans {OUTPUT_DIR}/ pour inspection visuelle.")

    print(f"\n  Problèmes détectés :")
    for diag, count in df["probleme_detecte"].value_counts().items():
        print(f"    {count:3d} PDF(s) → {diag}")
    print(f"{'='*55}\n")


if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
from PIL import Image


# ─────────────────────────────────────────────
# PARAMÈTRES — ajustez si la détection est imprécise
# ─────────────────────────────────────────────

# On ne cherche le footer que dans la moitié basse de l'image
SKIP_TOP_RATIO = 0.5

# Une ligne est considérée "blanche" si 92% de ses pixels sont > 240
SEUIL_LIGNE_BLANCHE = 0.92

# Nombre minimum de lignes blanches consécutives pour valider la séparation
MIN_LIGNES_BLANCHES = 3

# Marge de sécurité au-dessus de la coupure (pixels)
MARGE_PX = 5

# Pour confirmer qu'il y a bien un footer : % minimum de pixels sombres
# (< 180) sous la séparation. En dessous de ce seuil = juste une marge blanche
SEUIL_CONTENU_FOOTER = 0.01  # 1% de pixels sombres minimum sous la coupure


# ─────────────────────────────────────────────
# FONCTION PRINCIPALE
# ─────────────────────────────────────────────

def crop_footer_si_present(img: Image.Image) -> tuple[Image.Image, bool]:
    """
    Détecte et supprime le footer en bas d'une carte grise espagnole.

    Le footer (codes-barres, texte DGT, QR code) est séparé du tableau
    principal par une zone de lignes blanches horizontales.

    Robustesse :
      - Si aucune séparation n'est trouvée → image originale retournée
      - Si une séparation est trouvée MAIS qu'il n'y a pas de contenu
        réel sous elle (juste une marge blanche) → image originale retournée
      - Seulement si séparation + contenu réel dessous → crop effectué

    Args:
        img : Image PIL issue de la conversion PDF → PNG

    Returns:
        (image_résultante, footer_détecté)
        - image_résultante : Image PIL sans le footer, ou originale si pas de footer
        - footer_détecté   : True si un footer a été trouvé et supprimé

    Intégration dans votre pipeline :
        images = convert_from_path(pdf_path, dpi=300, first_page=1, last_page=1)
        img = images[0]
        img, footer_detecte = crop_footer_si_present(img)
        # → encoder img en base64 et envoyer à l'OCR comme d'habitude
    """
    gray = np.array(img.convert("L"), dtype=np.float32)
    h, w = gray.shape

    # Zone de recherche : moitié basse uniquement
    debut_recherche = int(h * SKIP_TOP_RATIO)

    # Ratio de pixels clairs (> 240) pour chaque ligne horizontale
    ratio_blanc_par_ligne = (gray > 240).mean(axis=1)

    # ── Étape 1 : chercher une zone de lignes blanches consécutives ───────
    consecutives = 0
    y_separation = None

    for y in range(h - 1, debut_recherche, -1):
        if ratio_blanc_par_ligne[y] >= SEUIL_LIGNE_BLANCHE:
            consecutives += 1
            if consecutives >= MIN_LIGNES_BLANCHES:
                y_separation = y - consecutives + 1
                break
        else:
            consecutives = 0

    # Aucune séparation trouvée → pas de footer
    if y_separation is None:
        return img, False

    # ── Étape 2 : vérifier qu'il y a du contenu réel SOUS la séparation ──
    # (distingue un vrai footer d'une simple marge blanche en bas)
    zone_sous_separation = gray[y_separation:, :]
    ratio_pixels_sombres = (zone_sous_separation < 180).mean()

    if ratio_pixels_sombres < SEUIL_CONTENU_FOOTER:
        # Pas assez de contenu sous la séparation → c'est juste une marge
        return img, False

    # ── Étape 3 : crop validé ─────────────────────────────────────────────
    y_crop = max(0, y_separation - MARGE_PX)
    img_croppee = img.crop((0, 0, w, y_crop))

    return img_croppee, True

In [ ]:
from crop_footer import crop_footer_si_present

# Votre pipeline actuel
images = convert_from_path(pdf_path, dpi=300, first_page=1, last_page=1)
img = images[0]

# ← ajout : crop du footer si présent
img, footer_detecte = crop_footer_si_present(img)

# Suite inchangée
buf = io.BytesIO()
img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
# → envoyer b64 à Mistral OCR

In [ ]:
"""
APPROCHE B — Sauvegarde des PNGs + crop footer automatique
===========================================================
Reproduit EXACTEMENT votre pipeline actuel (PDF → PNG 300 DPI → base64)
avec en plus la détection et suppression du footer avant l'OCR.

Pour chaque PDF, génère dans approche_b_pngs/<nom_pdf>/ :
  - page_original.png   : PNG brut sorti de pdf2image (avant crop)
  - page_cropee.png     : PNG après suppression du footer (envoyé à l'OCR)
                          (absent si pas de footer détecté)
  - page_comparaison.png: image côte à côte avant / après avec ligne rouge
                          (absent si pas de footer détecté)
  - ocr_reponse.txt     : réponse brute de Mistral OCR
  - info.txt            : log complet de toutes les étapes

Produit aussi :
  - approche_b_rapport.csv : tableau récapitulatif de tous les PDFs

Dépendances :
    pip install pdf2image pillow pandas requests numpy

Usage :
    python approche_b.py
"""

import base64
import io
import time
import numpy as np
import requests
from pathlib import Path

import pandas as pd
from pdf2image import convert_from_path
from PIL import Image, ImageStat, ImageDraw, ImageFont

# ─────────────────────────────────────────────
# ⚙️  CONFIGURATION — ADAPTEZ CES VALEURS
# ─────────────────────────────────────────────

# Dossier contenant tous les PDFs
PDF_DIR = Path("/chemin/vers/dossier/pdfs")

# Fichier texte avec les noms des PDFs échoués (un nom par ligne, ex: doc123.pdf)
FAILED_LIST = Path("/chemin/vers/liste_73_noms.txt")

# Mettre False pour appeler Mistral OCR après la conversion PNG
# Mettre True pour inspecter les PNGs uniquement, sans appel API (plus rapide)
NO_OCR = False

MISTRAL_OCR_URL = "http://votre-serveur-interne/v1/ocr"
MISTRAL_API_KEY  = "votre-cle-api"
REQUEST_TIMEOUT  = 60
DPI              = 300

MIN_OCR_CHARS = 50

# Fichiers de sortie
OUTPUT_DIR     = Path("approche_b_pngs")
OUTPUT_RAPPORT = Path("approche_b_rapport.csv")

# ─────────────────────────────────────────────
# PARAMÈTRES CROP FOOTER
# ─────────────────────────────────────────────

SKIP_TOP_RATIO       = 0.5    # chercher footer uniquement dans la moitié basse
SEUIL_LIGNE_BLANCHE  = 0.92   # 92% pixels > 240 pour considérer une ligne blanche
MIN_LIGNES_BLANCHES  = 3      # nb min de lignes blanches consécutives
MARGE_PX             = 5      # pixels de marge au-dessus de la coupure
SEUIL_CONTENU_FOOTER = 0.01   # 1% pixels sombres sous la séparation min


# ─────────────────────────────────────────────
# CROP FOOTER
# ─────────────────────────────────────────────

def crop_footer_si_present(img: Image.Image) -> tuple:
    """
    Détecte et supprime le footer si présent.
    Retourne (image_résultante, footer_détecté, y_separation)
    """
    gray = np.array(img.convert("L"), dtype=np.float32)
    h, w = gray.shape

    debut_recherche      = int(h * SKIP_TOP_RATIO)
    ratio_blanc_par_ligne = (gray > 240).mean(axis=1)

    consecutives = 0
    y_separation = None
    for y in range(h - 1, debut_recherche, -1):
        if ratio_blanc_par_ligne[y] >= SEUIL_LIGNE_BLANCHE:
            consecutives += 1
            if consecutives >= MIN_LIGNES_BLANCHES:
                y_separation = y - consecutives + 1
                break
        else:
            consecutives = 0

    if y_separation is None:
        return img, False, None

    zone_sous = gray[y_separation:, :]
    ratio_sombres = (zone_sous < 180).mean()

    if ratio_sombres < SEUIL_CONTENU_FOOTER:
        return img, False, None

    y_crop      = max(0, y_separation - MARGE_PX)
    img_croppee = img.crop((0, 0, w, y_crop))
    return img_croppee, True, y_separation


# ─────────────────────────────────────────────
# IMAGE DE COMPARAISON AVANT / APRÈS
# ─────────────────────────────────────────────

def creer_image_comparaison(img_avant: Image.Image,
                             img_apres: Image.Image,
                             y_separation: int) -> Image.Image:
    """
    Crée une image côte à côte : original (avec ligne rouge) | cropée.
    Réduite à 50% pour rester lisible sans être trop lourde.
    """
    scale  = 0.5
    w_orig = int(img_avant.width  * scale)
    h_orig = int(img_avant.height * scale)
    w_crop = int(img_apres.width  * scale)
    h_crop = int(img_apres.height * scale)

    img_avant_r = img_avant.resize((w_orig, h_orig), Image.LANCZOS)
    img_apres_r = img_apres.resize((w_crop, h_crop), Image.LANCZOS)

    # Ligne rouge sur l'original (position y scalée)
    avant_avec_ligne = img_avant_r.copy()
    draw = ImageDraw.Draw(avant_avec_ligne)
    y_ligne = int(y_separation * scale)
    draw.line([(0, y_ligne), (w_orig, y_ligne)], fill=(220, 38, 38), width=3)
    draw.text((8, max(0, y_ligne - 18)), "coupure", fill=(220, 38, 38))

    # Fond blanc pour assembler les deux
    sep    = 20
    h_max  = max(h_orig, h_crop)
    canvas = Image.new("RGB", (w_orig + sep + w_crop, h_max + 40), (255, 255, 255))

    # Labels en haut
    draw_c = ImageDraw.Draw(canvas)
    draw_c.text((w_orig // 2 - 40,  8), "AVANT (original)", fill=(100, 100, 100))
    draw_c.text((w_orig + sep + w_crop // 2 - 35, 8), "APRÈS (envoyé OCR)", fill=(15, 110, 86))

    # Coller les deux images
    canvas.paste(avant_avec_ligne, (0,          30))
    canvas.paste(img_apres_r,      (w_orig + sep, 30))

    # Ligne verticale séparatrice
    draw_c.line([(w_orig + sep // 2, 0), (w_orig + sep // 2, h_max + 40)],
                fill=(200, 200, 200), width=1)

    return canvas


# ─────────────────────────────────────────────
# CONVERSION PDF → PNG
# ─────────────────────────────────────────────

def convertir_pdf_png(pdf_path: str) -> dict:
    result = {"ok": False, "image": None, "erreur": None}
    try:
        images = convert_from_path(pdf_path, dpi=DPI, first_page=1, last_page=1, fmt="PNG")
        if not images:
            result["erreur"] = "pdf2image a retourné une liste vide"
            return result
        result["image"] = images[0]
        result["ok"]    = True
    except Exception as e:
        result["erreur"] = str(e)
    return result


# ─────────────────────────────────────────────
# INSPECTION PNG
# ─────────────────────────────────────────────

def inspecter_png(img: Image.Image) -> dict:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    taille_kb  = round(len(buf.getvalue()) / 1024, 2)
    gray       = img.convert("L")
    stat       = ImageStat.Stat(gray)
    luminosite = round(stat.mean[0], 2)
    ecart_type = round(stat.stddev[0], 2)
    return {
        "largeur_px":          img.width,
        "hauteur_px":          img.height,
        "taille_kb":           taille_kb,
        "luminosite":          luminosite,
        "ecart_type":          ecart_type,
        "est_blanc":           luminosite > 248,
        "est_noir":            luminosite < 5,
        "faible_contraste":    ecart_type < 10,
        "probablement_tourne": img.width < img.height,
    }


# ─────────────────────────────────────────────
# ENCODAGE BASE64
# ─────────────────────────────────────────────

def encoder_base64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


# ─────────────────────────────────────────────
# APPEL MISTRAL OCR
# ─────────────────────────────────────────────

def appeler_mistral_ocr(b64: str, filename: str) -> dict:
    result = {"ok": False, "http_status": None, "temps_s": None,
              "texte": "", "nb_chars": 0, "erreur": None}
    # ⚠️  Adaptez le payload à votre API interne
    payload = {"image": b64, "filename": filename}
    headers = {"Authorization": f"Bearer {MISTRAL_API_KEY}"}
    try:
        t0   = time.time()
        resp = requests.post(MISTRAL_OCR_URL, json=payload,
                             headers=headers, timeout=REQUEST_TIMEOUT)
        result["temps_s"]     = round(time.time() - t0, 2)
        result["http_status"] = resp.status_code
        resp.raise_for_status()
        data  = resp.json()
        texte = (data.get("text") or data.get("content") or data.get("result") or "").strip()
        result["texte"]    = texte
        result["nb_chars"] = len(texte)
        result["ok"]       = len(texte) >= MIN_OCR_CHARS
    except requests.exceptions.Timeout:
        result["erreur"] = f"Timeout ({REQUEST_TIMEOUT}s)"
    except Exception as e:
        result["erreur"] = str(e)
    return result


# ─────────────────────────────────────────────
# PIPELINE COMPLET POUR 1 PDF
# ─────────────────────────────────────────────

def traiter_un_pdf(pdf_path: str) -> dict:
    nom     = Path(pdf_path).name
    dossier = OUTPUT_DIR / Path(pdf_path).stem
    dossier.mkdir(parents=True, exist_ok=True)

    logs = []  # log lisible dans info.txt
    def log(msg: str, niveau: str = "INFO"):
        symbole = {"INFO": "  ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌", "CROP": "✂️ "}
        ligne = f"{symbole.get(niveau, '  ')} {msg}"
        logs.append(ligne)
        print(f"    {ligne}")

    row = {
        "filename":              nom,
        "dossier":               str(dossier),
        # Conversion
        "conversion_ok":         False,
        "conversion_erreur":     None,
        # PNG original
        "png_largeur_px":        None,
        "png_hauteur_px":        None,
        "png_taille_kb":         None,
        "png_luminosite":        None,
        "png_ecart_type":        None,
        "png_est_blanc":         None,
        "png_est_noir":          None,
        "png_faible_contraste":  None,
        "png_tourne_suspect":    None,
        # Crop footer
        "footer_detecte":        False,
        "footer_y_separation":   None,
        "footer_hauteur_avant":  None,
        "footer_hauteur_apres":  None,
        "footer_pct_supprime":   None,
        # OCR
        "ocr_appele":            not NO_OCR,
        "ocr_ok":                None,
        "ocr_http_status":       None,
        "ocr_temps_s":           None,
        "ocr_nb_chars":          None,
        "ocr_erreur":            None,
        "probleme_detecte":      "—",
    }

    # ── ÉTAPE 1 : Conversion PDF → PNG ────────────────────────────────────
    log("ÉTAPE 1 — Conversion PDF → PNG")
    conv = convertir_pdf_png(pdf_path)
    row["conversion_ok"]     = conv["ok"]
    row["conversion_erreur"] = conv["erreur"]

    if not conv["ok"]:
        log(f"Conversion échouée : {conv['erreur']}", "ERR")
        row["probleme_detecte"] = f"CONVERSION ÉCHOUÉE : {conv['erreur']}"
        (dossier / "info.txt").write_text("\n".join(logs), encoding="utf-8")
        return row

    img_original = conv["image"]
    img_original.save(dossier / "page_original.png", format="PNG")
    log(f"PNG produit : {img_original.width}x{img_original.height}px", "OK")

    # ── ÉTAPE 2 : Inspection du PNG original ─────────────────────────────
    log("ÉTAPE 2 — Inspection du PNG original")
    info = inspecter_png(img_original)
    row.update({
        "png_largeur_px":       info["largeur_px"],
        "png_hauteur_px":       info["hauteur_px"],
        "png_taille_kb":        info["taille_kb"],
        "png_luminosite":       info["luminosite"],
        "png_ecart_type":       info["ecart_type"],
        "png_est_blanc":        info["est_blanc"],
        "png_est_noir":         info["est_noir"],
        "png_faible_contraste": info["faible_contraste"],
        "png_tourne_suspect":   info["probablement_tourne"],
    })
    log(f"Taille : {info['taille_kb']} KB  |  Luminosité : {info['luminosite']}/255  |  Écart-type : {info['ecart_type']}")

    warnings = []
    if info["est_blanc"]:           warnings.append("PAGE BLANCHE")
    if info["est_noir"]:            warnings.append("PAGE NOIRE")
    if info["faible_contraste"]:    warnings.append("CONTRASTE FAIBLE")
    if info["probablement_tourne"]: warnings.append("ROTATION SUSPECTE")
    if info["taille_kb"] < 50:      warnings.append(f"PNG TRÈS PETIT ({info['taille_kb']} KB)")

    if warnings:
        log(f"Anomalies détectées : {' | '.join(warnings)}", "WARN")
    else:
        log("PNG visuellement correct", "OK")

    # ── ÉTAPE 3 : Détection et crop du footer ─────────────────────────────
    log("ÉTAPE 3 — Détection footer")
    img_a_envoyer, footer_detecte, y_sep = crop_footer_si_present(img_original)

    row["footer_detecte"]      = footer_detecte
    row["footer_y_separation"] = y_sep
    row["footer_hauteur_avant"] = img_original.height

    if footer_detecte:
        pct = round((1 - img_a_envoyer.height / img_original.height) * 100, 1)
        row["footer_hauteur_apres"] = img_a_envoyer.height
        row["footer_pct_supprime"]  = pct

        log(f"Footer détecté à y={y_sep}px", "CROP")
        log(f"Hauteur avant : {img_original.height}px → après : {img_a_envoyer.height}px (-{pct}%)", "CROP")

        # Sauvegarde PNG cropé
        img_a_envoyer.save(dossier / "page_cropee.png", format="PNG")
        log("page_cropee.png sauvegardé", "OK")

        # Image de comparaison avant / après
        comparaison = creer_image_comparaison(img_original, img_a_envoyer, y_sep)
        comparaison.save(dossier / "page_comparaison.png", format="PNG")
        log("page_comparaison.png sauvegardé (avant/après côte à côte)", "OK")
    else:
        row["footer_hauteur_apres"] = img_original.height
        row["footer_pct_supprime"]  = 0.0
        log("Aucun footer détecté → image originale envoyée à l'OCR")

    # ── ÉTAPE 4 : Encodage base64 ─────────────────────────────────────────
    log("ÉTAPE 4 — Encodage base64")
    b64    = encoder_base64(img_a_envoyer)
    b64_kb = round(len(b64) / 1024, 1)
    log(f"base64 : {b64_kb} KB ({len(b64)} chars)")

    # ── ÉTAPE 5 : Appel Mistral OCR ───────────────────────────────────────
    if not NO_OCR:
        log("ÉTAPE 5 — Appel Mistral OCR")
        ocr = appeler_mistral_ocr(b64, nom)
        row.update({
            "ocr_ok":          ocr["ok"],
            "ocr_http_status": ocr["http_status"],
            "ocr_temps_s":     ocr["temps_s"],
            "ocr_nb_chars":    ocr["nb_chars"],
            "ocr_erreur":      ocr["erreur"],
        })

        (dossier / "ocr_reponse.txt").write_text(
            ocr["texte"] or f"[VIDE — erreur: {ocr['erreur']}]",
            encoding="utf-8"
        )

        if not ocr["ok"]:
            diag = ocr["erreur"] or f"Réponse trop courte ({ocr['nb_chars']} chars)"
            log(f"OCR échoué : {diag}", "ERR")
            if not warnings:
                row["probleme_detecte"] = f"OCR ÉCHOUÉ : {diag}"
        else:
            log(f"OCR OK : {ocr['nb_chars']} chars en {ocr['temps_s']}s", "OK")
            if not warnings:
                row["probleme_detecte"] = "AUCUN — pipeline OK"
    else:
        log("ÉTAPE 5 — OCR non appelé (NO_OCR = True)")

    if warnings and not row["probleme_detecte"] != "—":
        row["probleme_detecte"] = " | ".join(warnings)

    # ── Sauvegarde du log ─────────────────────────────────────────────────
    log_txt  = f"{'='*50}\n"
    log_txt += f"FICHIER : {nom}\n"
    log_txt += f"{'='*50}\n\n"
    log_txt += "\n".join(logs)
    log_txt += f"\n\n{'─'*50}\n"
    log_txt += f"Fichiers générés dans : {dossier}/\n"
    log_txt += f"  page_original.png     : PNG brut (avant crop)\n"
    if footer_detecte:
        log_txt += f"  page_cropee.png       : PNG sans footer (envoyé à l'OCR)\n"
        log_txt += f"  page_comparaison.png  : avant / après côte à côte\n"
    log_txt += f"  ocr_reponse.txt       : réponse brute Mistral OCR\n"

    (dossier / "info.txt").write_text(log_txt, encoding="utf-8")
    return row


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    if not PDF_DIR.is_dir():
        print(f"❌ PDF_DIR introuvable : {PDF_DIR}")
        return
    if not FAILED_LIST.exists():
        print(f"❌ FAILED_LIST introuvable : {FAILED_LIST}")
        return

    OUTPUT_DIR.mkdir(exist_ok=True)

    noms = [l.strip() for l in FAILED_LIST.read_text(encoding="utf-8").splitlines() if l.strip()]

    chemins      = []
    introuvables = []
    for nom in noms:
        chemin = PDF_DIR / nom
        if not chemin.exists():
            print(f"⚠️  Fichier introuvable : {chemin}")
            introuvables.append(nom)
        chemins.append(str(chemin))

    mode = "inspection PNG uniquement" if NO_OCR else "PNG + crop footer + appel OCR"
    print(f"\n{'='*55}")
    print(f"  APPROCHE B — PNG + crop footer")
    print(f"  Dossier PDFs    : {PDF_DIR}")
    print(f"  PDFs à traiter  : {len(chemins)}")
    if introuvables:
        print(f"  ⚠️  Introuvables  : {len(introuvables)}")
    print(f"  Mode            : {mode}")
    print(f"  Dossier sortie  : {OUTPUT_DIR}/")
    print(f"{'='*55}\n")

    rows = []
    for i, chemin in enumerate(chemins, 1):
        print(f"\n[{i:3}/{len(chemins)}] {Path(chemin).name}")
        print(f"  {'─'*50}")
        row = traiter_un_pdf(chemin)
        rows.append(row)

    # ── Rapport CSV ───────────────────────────────────────────────────────
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_RAPPORT, index=False, encoding="utf-8-sig")

    # ── Synthèse console ──────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  SYNTHÈSE APPROCHE B")
    print(f"{'='*55}")
    print(f"  Footers détectés et supprimés : {df['footer_detecte'].sum()}/{len(df)}")
    print(f"  PNGs blancs/noirs             : {(df['png_est_blanc'] | df['png_est_noir']).sum()}")
    print(f"  Faible contraste              : {df['png_faible_contraste'].sum()}")
    print(f"  Rotation suspecte             : {df['png_tourne_suspect'].sum()}")
    print(f"  Erreurs conversion            : {(~df['conversion_ok']).sum()}")

    if df["footer_detecte"].sum() > 0:
        pcts = df[df["footer_detecte"]]["footer_pct_supprime"]
        print(f"  % image supprimé (footer)     : min={pcts.min():.1f}%  max={pcts.max():.1f}%  moy={pcts.mean():.1f}%")

    if not NO_OCR:
        print(f"  OCR échoués                   : {(~df['ocr_ok'].fillna(True)).sum()}")
        n_footer_ok = ((df['footer_detecte']) & (df['ocr_ok'].fillna(False))).sum()
        n_footer_ko = ((df['footer_detecte']) & (~df['ocr_ok'].fillna(True))).sum()
        print(f"  Avec footer → OCR OK          : {n_footer_ok}")
        print(f"  Avec footer → OCR encore KO   : {n_footer_ko}")

    print(f"\n  Rapport CSV  : {OUTPUT_RAPPORT}")
    print(f"  Dossiers     : {OUTPUT_DIR}/<nom_pdf>/")
    print(f"     ├── page_original.png      (PNG brut avant crop)")
    print(f"     ├── page_cropee.png        (PNG envoyé à l'OCR — si footer)")
    print(f"     ├── page_comparaison.png   (avant / après côte à côte — si footer)")
    print(f"     ├── ocr_reponse.txt        (réponse brute Mistral OCR)")
    print(f"     └── info.txt               (log complet étape par étape)")

    print(f"\n  Problèmes détectés :")
    for diag, count in df["probleme_detecte"].value_counts().items():
        print(f"    {count:3d} PDF(s) → {diag}")
    print(f"{'='*55}\n")


if __name__ == "__main__":
    main()

In [ ]:
from crop_footer import crop_footer_si_present  # en haut du fichier

# Votre pipeline avec crop intégré
images = convert_from_path(pdf_path, dpi=300, first_page=1, last_page=1)
img = images[0]

# ← LA SEULE LIGNE À AJOUTER
img, _ = crop_footer_si_present(img)

buf = io.BytesIO()
img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

# → appel OCR avec b64, rien d'autre ne change

In [ ]:
#ooter
#supprimé sur {p
#df_path}")

## “””
crop_above_table.py

Détecte le contour supérieur d’un tableau dans une image de carte grise
espagnole (MDF) et supprime tout ce qui se trouve au-dessus.

Usage:
python crop_above_table.py input.jpg           -> génère input_cropped.jpg
python crop_above_table.py input.jpg -o out.jpg
python crop_above_table.py input.jpg –debug   -> affiche les étapes intermédiaires

Dépendances:
pip install opencv-python-headless numpy
“””

import cv2
import numpy as np
import argparse
import os
import sys

def detect_table_top(image_path: str, debug: bool = False) -> int:
“””
Détecte la coordonnée Y du bord supérieur du tableau principal.

```
Stratégie :
1. Pré-traitement : niveaux de gris + flou léger
2. Détection de lignes horizontales via morphologie (érosion/dilatation)
3. La PREMIERE ligne horizontale longue = bord supérieur du tableau
4. Fallback : détection de contours si la morphologie échoue

Returns:
    y_top (int) : coordonnée Y du bord supérieur du tableau
"""
img = cv2.imread(image_path)
if img is None:
    raise FileNotFoundError(f"Impossible de lire l'image : {image_path}")

h, w = img.shape[:2]
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# --- Étape 1 : binarisation adaptative ---
blur = cv2.GaussianBlur(gray, (3, 3), 0)
binary = cv2.adaptiveThreshold(
    blur, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    15, 4
)

# --- Étape 2 : extraction des lignes horizontales ---
# Noyau large = seules les lignes qui s'étendent sur ≥ 40% de la largeur survivent
min_line_width = int(w * 0.40)
kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (min_line_width, 1))
horizontal_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_h)

# Dilate légèrement pour connecter les segments proches
dilate_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3))
horizontal_lines = cv2.dilate(horizontal_lines, dilate_kernel)

if debug:
    cv2.imwrite("debug_binary.jpg", binary)
    cv2.imwrite("debug_hlines.jpg", horizontal_lines)

# --- Étape 3 : trouver la première ligne horizontale ---
# On somme chaque ligne de pixels : une forte somme = ligne détectée
row_sums = horizontal_lines.sum(axis=1)

# Seuil : la ligne doit couvrir au moins 30% de la largeur
threshold = w * 0.30 * 255

candidate_rows = np.where(row_sums > threshold)[0]

if len(candidate_rows) > 0:
    y_top = int(candidate_rows[0])
    print(f"✓ Bord supérieur du tableau détecté à Y = {y_top}px (méthode: lignes horizontales)")
    return y_top

# --- Fallback : détection de contours rectangulaires ---
print("⚠ Fallback : recherche de contours rectangulaires...")
edges = cv2.Canny(blur, 50, 150)
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

best_y = None
best_score = 0

for cnt in contours:
    x, y, cw, ch = cv2.boundingRect(cnt)
    # On cherche un rectangle large (> 50% de w) et pas trop petit (> 5% de h)
    if cw > w * 0.50 and ch > h * 0.05:
        score = cw * ch
        if score > best_score:
            best_score = score
            best_y = y

if best_y is not None:
    print(f"✓ Bord supérieur détecté à Y = {best_y}px (méthode: contours)")
    return best_y

# Dernier recours : 20% depuis le haut
fallback_y = int(h * 0.20)
print(f"⚠ Aucun tableau trouvé. Crop par défaut à Y = {fallback_y}px (20% depuis le haut)")
return fallback_y
```

def crop_above_table(image_path: str, output_path: str = None, debug: bool = False,
margin: int = 2) -> str:
“””
Coupe tout ce qui est au-dessus du bord supérieur du tableau.

```
Args:
    image_path  : chemin de l'image source
    output_path : chemin de sortie (par défaut : <nom>_cropped.<ext>)
    debug       : enregistre les images intermédiaires
    margin      : pixels de marge à garder au-dessus de la ligne (défaut 2)

Returns:
    output_path (str) : chemin de l'image croppée
"""
img = cv2.imread(image_path)
if img is None:
    raise FileNotFoundError(f"Impossible de lire : {image_path}")

y_top = detect_table_top(image_path, debug=debug)

# On garde une petite marge pour ne pas couper le filet supérieur du tableau
y_crop = max(0, y_top - margin)

cropped = img[y_crop:, :]

if output_path is None:
    base, ext = os.path.splitext(image_path)
    output_path = f"{base}_cropped{ext}"

cv2.imwrite(output_path, cropped)

h_orig = img.shape[0]
h_crop = cropped.shape[0]
removed = y_crop
print(f"✓ Supprimé : {removed}px en haut ({removed/h_orig*100:.1f}% de la hauteur originale)")
print(f"✓ Image croppée : {h_crop}px de hauteur → sauvegardée dans : {output_path}")

if debug:
    debug_img = img.copy()
    cv2.line(debug_img, (0, y_top), (img.shape[1], y_top), (0, 0, 255), 3)
    cv2.imwrite("debug_detection.jpg", debug_img)
    print("✓ Image de debug (ligne rouge = bord détecté) : debug_detection.jpg")

return output_path
```

# ──────────────────────────────────────────────

# CLI

# ──────────────────────────────────────────────

if **name** == “**main**”:
parser = argparse.ArgumentParser(
description=“Crop tout ce qui est au-dessus du tableau dans une carte grise espagnole.”
)
parser.add_argument(“input”, help=“Chemin de l’image source (JPG, PNG, TIFF…)”)
parser.add_argument(”-o”, “–output”, default=None,
help=“Chemin de l’image de sortie (défaut : <nom>_cropped.<ext>)”)
parser.add_argument(”–debug”, action=“store_true”,
help=“Enregistre les images intermédiaires de diagnostic”)
parser.add_argument(”–margin”, type=int, default=2,
help=“Pixels de marge à conserver au-dessus du tableau (défaut : 2)”)

```
args = parser.parse_args()

if not os.path.exists(args.input):
    print(f"Erreur : fichier introuvable → {args.input}", file=sys.stderr)
    sys.exit(1)

try:
    out = crop_above_table(args.input, args.output, debug=args.debug, margin=args.margin)
    print(f"\n✅ Terminé ! Image croppée : {out}")
except Exception as e:
    print(f"Erreur : {e}", file=sys.stderr)
    sys.exit(1)
```